In [ ]:
# ruff: noqa: F401
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from ipywidgets.widgets import Button, Dropdown, HBox, Output, VBox, interact
from plot_utils import *
from scipy import stats

In [ ]:
RESULTS = {
    "CA_P": load_results("ca_housing"),
    "CA_I": load_results("ca_housing_income"),
    "UK": load_results("housing"),
}

In [ ]:
ALGORITHM = Dropdown(
    options=["mondrian", "kmember", "incognito"],
    value="mondrian",
    description="Algorithm:",
)
DATASET = Dropdown(
    options=[
        ("CA_HOUSING (price)", "CA_P"),
        ("CA_HOUSING (income)", "CA_I"),
        ("UK_HOUSING", "UK"),
    ],
    value="CA_P",
    description="Dataset:",
)
X_AXIS = Dropdown(
    options=[
        ("K", "K"),
        ("NCP", "UT_NCP"),
        ("CAVG", "UT_CAVG"),
        ("DM", "UT_DM"),
        ("Penalty_REG", "UT_RM"),
        ("Penalty_BIN", "UT_CM_BIN"),
        ("Penalty_MUL", "UT_CM_MUL"),
    ],
    value="UT_NCP",
    description="x-axis:",
)
PLOT_TYPE = Dropdown(
    options=[
        ("1 algorithm & 1 dataset", 0),
        ("1 algorithm & all datasets", 1),
        ("all algorithms & 1 dataset", 2),
    ],
    value=0,
    description="Plot type:",
)
FIT_TYPE = Dropdown(
    options=["linear", "logarithmic", "exponential", "power"],
    value="linear",
    description="Trendline:",
)
K_SELECT = Dropdown(
    options=["all"] + K,
    value="all",
    description="Select k:",
)

In [ ]:
def make_y_axis(x: str, last_value=None):
    if "UT_RM" in x:
        return Dropdown(
            options=[("Error_REG", "ML_REG_RMSE")],
            value="ML_REG_RMSE",
            description="y-axis:",
        )
    if "UT_CM_BIN" in x:
        return Dropdown(
            options=[("Error_BIN", "ML_CLS_BIN_F1_LOSS")],
            value="ML_CLS_BIN_F1_LOSS",
            description="y-axis:",
        )
    if "UT_CM_MUL" in x:
        return Dropdown(
            options=[("Error_MUL", "ML_CLS_MUL_F1_LOSS")],
            value="ML_CLS_MUL_F1_LOSS",
            description="y-axis:",
        )
    return Dropdown(
        options=[
            ("Error_REG", "ML_REG_RMSE"),
            ("Error_BIN", "ML_CLS_BIN_F1_LOSS"),
            ("Error_MUL", "ML_CLS_MUL_F1_LOSS"),
            ("Vulnerability", "VUL_DBRL"),
        ],
        value=last_value if last_value is not None else "ML_CLS_BIN_F1_LOSS",
        description="y-axis:",
    )

In [ ]:
def get_x_y_single(algo: str, dataset, x: str, y: str, k: str | int):
    if k == "all":
        X = [dataset[algo][x].tolist()]
        Y = [dataset[algo][y].tolist()]
    else:
        pos_k = K.index(k)
        X = [dataset[algo][x].tolist()[pos_k : pos_k + 1]]
        Y = [dataset[algo][y].tolist()[pos_k : pos_k + 1]]
    return (X, Y)

In [ ]:
def get_x_y_all_dataset(algo: str, x: str, y: str, k: str | int):
    X = []
    Y = []
    for dataset in [v for _, v in RESULTS.items()]:
        if k == "all":
            X.append(dataset[algo][x].tolist())
            Y.append(dataset[algo][y].tolist())
        else:
            pos_k = K.index(k)
            X.append(dataset[algo][x].tolist()[pos_k : pos_k + 1])
            Y.append(dataset[algo][y].tolist()[pos_k : pos_k + 1])
    return (X, Y)

In [ ]:
def get_x_y_all_algo(dataset, x: str, y: str, k: str | int):
    X = []
    Y = []
    algos = ["mondrian", "kmember", "incognito"]
    for algo in algos:
        if k == "all":
            X.append(dataset[algo][x].tolist())
            Y.append(dataset[algo][y].tolist())
        else:
            pos_k = K.index(k)
            X.append(dataset[algo][x].tolist()[pos_k : pos_k + 1])
            Y.append(dataset[algo][y].tolist()[pos_k : pos_k + 1])
    return (X, Y)

In [ ]:
def draw_trend_line(AX, X, Y, fit: str = "linear"):
    x_flatten = [item for sublist in X for item in sublist]
    x_trend = np.linspace(np.min(x_flatten), np.max(x_flatten), 100)

    y_flatten = [item for sublist in Y for item in sublist]

    if fit == "linear":
        res = stats.linregress(x_flatten, y_flatten)
        y_trend = res.slope * x_trend + res.intercept
    elif fit == "logarithmic":
        _x = np.log(x_flatten)
        y_trend = np.linspace(np.min(y_flatten), np.max(y_flatten), 100)
        res = stats.linregress(y_flatten, _x)
        x_trend = np.exp(res.intercept) * np.exp(y_trend * res.slope)
    elif fit == "exponential":
        _y = np.log(y_flatten)
        res = stats.linregress(x_flatten, _y)
        y_trend = np.exp(res.intercept) * np.exp(x_trend * res.slope)
    elif fit == "power":
        _x = np.log(x_flatten)
        _y = np.log(y_flatten)
        res = stats.linregress(_x, _y)
        y_trend = np.exp(res.intercept) * x_trend**res.slope

    AX.plot(x_trend, y_trend, c="white", lw=4.5)

    AX.plot(
        x_trend,
        y_trend,
        c="tab:green",
        lw=3,
        label=f"r^2={res.rvalue**2:.2f}, "
        + f"p={f'{res.pvalue:.2e}' if res.pvalue < 0.001 else f'{res.pvalue:.3f}'}",
    )

In [ ]:
def set_ax_lim(AX, x, y):
    if x not in ["UT_CAVG", "UT_DM"]:
        AX.set_xlim([0, 1])
        AX.set_xticks(np.arange(0, 1.1, 0.2))
    if y not in ["UT_CAVG", "UT_DM"]:
        AX.set_ylim([0, 1])
        AX.set_yticks(np.arange(0, 1.1, 0.2))

In [ ]:
@interact
def render(plot_type=PLOT_TYPE):
    Y_AXIS = make_y_axis(X_AXIS.value)
    if plot_type == 0:
        algo_dataset_selection = HBox([ALGORITHM, DATASET])
        K_SELECT.value = "all"
        k_n_fit = [FIT_TYPE]
    elif plot_type == 1:
        algo_dataset_selection = ALGORITHM
        k_n_fit = [FIT_TYPE, K_SELECT]
    elif plot_type == 2:
        algo_dataset_selection = DATASET
        k_n_fit = [FIT_TYPE, K_SELECT]

    out_plot = Output()
    btn = Button(description="Plot")

    def plot(b):
        FIG, AX = plt.subplots(figsize=(6, 4))
        plt.close(FIG)
        if plot_type == 0:
            X, Y = get_x_y_single(
                ALGORITHM.value,
                RESULTS[DATASET.value],
                X_AXIS.value,
                Y_AXIS.value,
                K_SELECT.value,
            )
        elif plot_type == 1:
            X, Y = get_x_y_all_dataset(
                ALGORITHM.value, X_AXIS.value, Y_AXIS.value, K_SELECT.value
            )
        elif plot_type == 2:
            X, Y = get_x_y_all_algo(
                RESULTS[DATASET.value], X_AXIS.value, Y_AXIS.value, K_SELECT.value
            )
        for i in range(len(X)):
            AX.scatter(
                X[i],
                Y[i],
                marker=CORR_SPECS[i]["marker"],
                edgecolors="k",
                linewidths=1,
                alpha=ALPHA,
                s=150,
                c=K if K_SELECT.value == "all" else [K_SELECT.value],
                cmap=CORR_SPECS[i]["cmap"],
                vmin=2,
                vmax=100,
            )

        AX.set_xlabel(X_AXIS.label)
        AX.set_ylabel(Y_AXIS.label)
        set_ax_lim(AX, X_AXIS.value, Y_AXIS.value)
        AX.grid()

        if X_AXIS.value != "K":
            draw_trend_line(AX, X, Y, FIT_TYPE.value)
            AX.legend()
        with out_plot:
            out_plot.clear_output(wait=True)
            display(FIG)
        return

    btn.on_click(plot)
    out_selection = Output()

    def on_x_change(change):
        nonlocal Y_AXIS
        Y_AXIS = make_y_axis(change["new"], Y_AXIS.value)
        with out_selection:
            out_selection.clear_output(wait=True)
            display(
                VBox(
                    [
                        algo_dataset_selection,
                        HBox([X_AXIS, Y_AXIS]),
                        HBox(k_n_fit),
                    ]
                )
            )

    X_AXIS.observe(on_x_change, names="value")
    X_AXIS.value = "UT_NCP"
    on_x_change({"new": "UT_NCP"})
    btn.click()
    display(VBox([out_selection, btn, out_plot]))